In [1]:
import pandas as pd
from vllm import LLM, SamplingParams
from tqdm import tqdm
import json
import os
import re
from typing import List, Dict, Any # Added missing type hint imports if not present globally

# ================= CONFIGURATION =================
# INPUT_FILES is now DATASET_CSV to match the logic below
DATASET_CSV = "data/paired_vuln_fixed.csv" 
OUTPUT_DIR = "data_llm_filter_output"
OUTPUT_FILE_ACCEPTED = "accepted_for_triage.csv"
OUTPUT_FILE_REJECTED = "rejected_by_auditor.csv"

# Model swapped to Qwen 32B for improved coding reasoning
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct" 
# =================================================

# 2. Initialize vLLM
print(f"--- Loading Model: {MODEL_NAME} ---")
# Set capacity higher than the maximum required input (8809)
# Using 16384 (16k) provides a safer buffer for future large inputs.
llm = LLM(model=MODEL_NAME, tensor_parallel_size=1, max_model_len=16384)
tokenizer = llm.get_tokenizer()

# Sampling parameters for strict, fast decisions
sampling_params = SamplingParams(
    temperature=0.05, 
    top_p=0.90, 
    max_tokens=512 
)


/home/tkavuru/.conda/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-20 17:10:04 [__init__.py:216] Automatically detected platform cuda.
--- Loading Model: Qwen/Qwen2.5-Coder-32B-Instruct ---
INFO 11-20 17:10:13 [utils.py:233] non-default args: {'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-32B-Instruct'}
INFO 11-20 17:10:13 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-20 17:10:13 [model.py:1510] Using max model len 16384


2025-11-20 17:10:14,826	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-20 17:10:14 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:15 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:15 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen2.5-Coder-32B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-Coder-32B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_c

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:00<00:11,  1.16it/s]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:01<00:11,  1.05it/s]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:02<00:10,  1.05it/s]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:03<00:09,  1.06it/s]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:04<00:08,  1.08it/s]
Loading safetensors checkpoint shards:  43% Completed | 6/14 [00:05<00:07,  1.05it/s]
Loading safetensors checkpoint shards:  50% Completed | 7/14 [00:06<00:06,  1.04it/s]
Loading safetensors checkpoint shards:  57% Completed | 8/14 [00:07<00:05,  1.04it/s]
Loading safetensors checkpoint shards:  64% Completed | 9/14 [00:08<00:04,  1.04it/s]
Loading safetensors checkpoint shards:  71% Completed | 10/14 [00:09<00:03,  1.04it/s]
Loading safetensors checkpoint shards:  79% Completed | 11/14

(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:35 [default_loader.py:267] Loading weights took 12.99 seconds
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:36 [gpu_model_runner.py:2653] Model loading took 61.0375 GiB and 13.739951 seconds
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:51 [backends.py:548] Using cache directory: /home/tkavuru/.cache/vllm/torch_compile_cache/41f1b2542d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:51 [backends.py:559] Dynamo bytecode transform time: 14.42 s
(EngineCore_DP0 pid=3955795) INFO 11-20 17:10:58 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 6.790 s
(EngineCore_DP0 pid=3955795) INFO 11-20 17:11:09 [monitor.py:34] torch.compile takes 14.42 s in total
(EngineCore_DP0 pid=3955795) INFO 11-20 17:11:10 [gpu_worker.py:298] Available KV cache memory: 8.56 GiB
(EngineCore_DP0 pid=3955795) INFO 11-20 17:11:10 [kv_cache_utils.py:1087] GPU KV cache size: 35,056 tokens
(E

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:09<00:00,  6.76it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.25it/s]


(EngineCore_DP0 pid=3955795) INFO 11-20 17:11:25 [gpu_model_runner.py:3480] Graph capturing finished in 15 secs, took 1.32 GiB
(EngineCore_DP0 pid=3955795) INFO 11-20 17:11:25 [core.py:210] init engine (profile, create kv cache, warmup model) took 49.47 seconds
INFO 11-20 17:11:27 [llm.py:306] Supported_tasks: ['generate']


In [3]:
def clean_json_output(text: str) -> Dict[str, Any] | None:
    """Robustly extracts and loads JSON."""
    try:
        text = text.strip()
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if match: text = match.group(1)
        
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end != -1:
            return json.loads(text[start:end])
        return None
    except Exception:
        return None

# --- CONFIGURATION FOR SAMPLING ---
# NOTE: Adjust SAMPLE_SIZE for verification (e.g., 50) or comment out for full run.
SAMPLE_SIZE = 100000 
# ----------------------------------

# Access globals loaded in the previous cell
global llm, tokenizer, DATASET_CSV, OUTPUT_DIR, OUTPUT_FILE_ACCEPTED, OUTPUT_FILE_REJECTED

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load Data (Assumes data is already paired)
if not os.path.exists(DATASET_CSV):
    print(f"FATAL: Input file {DATASET_CSV} not found.")

df_full = pd.read_csv(DATASET_CSV)

# Filter massive functions (using the vulnerable body length as the primary proxy)
df_clean = df_full[df_full['vuln_func_body'].str.len() <= 12000].copy()

dropped = len(df_full) - len(df_clean)
if dropped > 0: print(f"\n-> Dropped {dropped} functions based on max length.")

# 2. Apply Sampling Logic
if len(df_clean) > SAMPLE_SIZE:
    df = df_clean.sample(n=SAMPLE_SIZE).copy()
    print(f"--- Running verification sample of {SAMPLE_SIZE} pairs ---")
else:
    df = df_clean.copy()
    print(f"--- Running full set of {len(df)} pairs ---")

# 3. The Ruthless Auditor Prompt (Tuned for Low-Logic Rejection)
system_prompt = """You are a Ruthless Code Auditor.
Your job is to REJECT any C function that is not a perfect candidate for a simple, standalone unit test.

### CRITERIA FOR REJECTION (Hard Deny):
1.  **Complex Environment:** Uses kernel locks, hardware I/O, or complex file/network stacks (vfs_, sock_).
2.  **External Frameworks (Opaque Action):** The function's main action relies on an external, non-standard API (e.g., `ssl_connect`, `xmlParse`, `gss_init`, `mysql_query`). This requires guessing complex logic.
3.  **Language Contamination:** Contains C++ classes, templates, or operator overloading.
4.  **Trivial Value (Pure Glue):** The function is pure glue, meaning it performs **zero discernible logic** (e.g., just passes arguments or returns an external call result: `return helper_func(x);`).
5.  **Data Dependence:** The function accesses a struct field that requires complex nested definitions or type inference that is too ambiguous.

### CRITERIA FOR ACCEPTANCE:
1.  **High Logic Value:** Performs observable math, conditional parsing, or loop/array manipulation **within its own body**.
2.  **Mockable Dependencies:** Dependencies are limited to standard C, POSIX, or easily mockable resources (simple data structs, malloc/free).

Respond with this JSON format ONLY:
{
    "decision": "ACCEPT" or "REJECT",
    "reason": "Brief justification for the decision."
}
"""

# 4. Prepare Prompts and Run Batch Inference
prompts = []
for _, row in df.iterrows():
    # Pass both bodies to the LLM for comprehensive auditing
    vuln_code = row.get('vuln_func_body', '')
    fixed_code = row.get('fixed_func_body', '')
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"""Audit Pair:
        VULNERABLE: ```c\n{vuln_code}\n```
        FIXED: ```c\n{fixed_code}\n```
        """}
    ]
    prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

sampling_params = SamplingParams(temperature=0.05, max_tokens=512)
print(f"-> Sending {len(prompts)} pairs to LLM for filtering...")

outputs = llm.generate(prompts, sampling_params)

# 5. Parse and Save Artifacts
# 6. Parse and Filter Results (FIXED WITH NULL CHECK)




-> Dropped 434 functions based on max length.
--- Running full set of 9590 pairs ---
-> Sending 9590 pairs to LLM for filtering...


Processed prompts: 100%|██████████| 9590/9590 [1:22:10<00:00,  1.94it/s, est. speed input: 2917.86 toks/s, output: 139.37 toks/s]  


In [4]:
accepted_rows = []
rejected_rows = []

for i, output in enumerate(tqdm(outputs, desc="Processing LLM Decisions")):
    generated_text = output.outputs[0].text
    json_data = clean_json_output(generated_text)
    
    # Use .iloc[i] on the sampled DF to maintain alignment
    row_data = df.iloc[i].copy() 
    
    # --- CRITICAL FIX: CHECK FOR NONE TYPE ---
    if json_data is not None and json_data.get('decision') == 'ACCEPT':
        row_data['filter_reason'] = json_data.get('reason', 'N/A')
        accepted_rows.append(row_data)
    else:
        # Assigning a reason based on whether JSON was present or not
        reason_text = json_data.get('reason', 'JSON Parse Failure') if json_data else f'RAW OUTPUT ERROR: {generated_text[:100]}...'
        row_data['filter_reason'] = reason_text
        rejected_rows.append(row_data)

# 6. Save Artifacts
filtered_df = pd.DataFrame(accepted_rows)
rejected_df = pd.DataFrame(rejected_rows)

accepted_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_ACCEPTED)
rejected_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_REJECTED)

print("\n--- LLM FILTER RESULTS ---")
print(f"Total Audited: {len(df)}")
print(f"Accepted Count: {len(filtered_df)}")
print(f"Rejected Count: {len(rejected_df)}")

filtered_df.to_csv(accepted_path, index=False)
rejected_df.to_csv(rejected_path, index=False)
print(f"Saved accepted dataset sample to: {accepted_path}")

Processing LLM Decisions: 100%|██████████| 9590/9590 [00:03<00:00, 3190.80it/s]



--- LLM FILTER RESULTS ---
Total Audited: 9590
Accepted Count: 71
Rejected Count: 9519
Saved accepted dataset sample to: data_llm_filter_output/accepted_for_triage.csv


In [1]:
import pandas as pd
from vllm import LLM, SamplingParams
from tqdm import tqdm
import json
import os
import re


# ================= CONFIGURATION =================
# Old: INPUT_FILES = ["data_llm_pass_1/dataset_medium_verified.csv", ...]
INPUT_FILES = [
    "data_llm_filter_output/accepted_for_triage.csv" # New single filtered source
]
OUTPUT_DIR = "data_context" 
OUTPUT_FILE = "dataset_unified_with_context.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct" 
# =================================================

def clean_json_output(text):
    try:
        text = text.strip()
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if match: text = match.group(1)
        else:
            # Fallback cleanup for Qwen if it misses code blocks
            if text.startswith("```"): text = text.split("\n", 1)[1]
            if text.endswith("```"): text = text.rsplit("\n", 1)[0]
        return json.loads(text)
    except:
        return None

def extract_code_block(text):
    try:
        match = re.search(r'```c\s*(.*?)\s*```', text, re.DOTALL)
        if match: return match.group(1)
        if "#include" in text: return text
        return ""
    except:
        return ""

def infer_signature_hints(source_code: str, function_list: list) -> dict:
    """
    Python Helper: Scans code to find how functions are called and infers a safe C signature.
    This helps the Builder generate the correct function stub and avoids unnecessary variadic stubs.
    """
    hints = {}
    for func in function_list:
        # Search for: func_name ( ... )
        # Escape func name and handle whitespace
        pattern = r'\b' + re.escape(func) + r'\s*\((.*?)\)'
        match = re.search(pattern, source_code, re.DOTALL)
        
        if match:
            args_str = match.group(1).strip()
            
            if not args_str:
                # Case: foo()
                hints[func] = f"void {func}(void)"
            else:
                # Heuristic 1: Detect Variadic (printf-style) based on leading quote
                if args_str.startswith('"') or args_str.startswith('L"'):
                    hints[func] = f"int {func}(const char *fmt, ...)"
                else:
                    # Heuristic 2: Count commas to estimate arg count
                    # This is the safer path for non-variadic calls (e.g., foo(a, b, c))
                    arg_count = args_str.count(',') + 1
                    # Create generic signature: int foo(void *arg0, void *arg1, ...)
                    args = ", ".join([f"void *arg{i}" for i in range(arg_count)])
                    hints[func] = f"int {func}({args})"
        else:
            # Fallback: If no call is found (e.g., the function is only used in a macro), 
            # default to variadic safety (int func(...))
            hints[func] = f"int {func}(...)"
            
    return hints

/home/tkavuru/.conda/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-20 23:12:17 [__init__.py:216] Automatically detected platform cuda.


In [2]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

dfs = []
for f in INPUT_FILES:
    if os.path.exists(f): dfs.append(pd.read_csv(f))
full_df = pd.concat(dfs, ignore_index=True)
print(f"Total pairs to process: {len(full_df)}")

print(f"--- Loading Model: {MODEL_NAME} ---")
llm = LLM(model=MODEL_NAME, tensor_parallel_size=1, max_model_len=16384)
tokenizer = llm.get_tokenizer()

Total pairs to process: 71
--- Loading Model: Qwen/Qwen2.5-Coder-32B-Instruct ---
INFO 11-20 23:12:42 [utils.py:233] non-default args: {'max_model_len': 16384, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Coder-32B-Instruct'}
INFO 11-20 23:12:42 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-20 23:12:42 [model.py:1510] Using max model len 16384


2025-11-20 23:12:43,658	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-20 23:12:43 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3759800) INFO 11-20 23:12:44 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=3759800) INFO 11-20 23:12:44 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen2.5-Coder-32B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-Coder-32B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_c

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/14 [00:07<01:39,  7.68s/it]
Loading safetensors checkpoint shards:  14% Completed | 2/14 [00:15<01:32,  7.71s/it]
Loading safetensors checkpoint shards:  21% Completed | 3/14 [00:22<01:22,  7.54s/it]
Loading safetensors checkpoint shards:  29% Completed | 4/14 [00:30<01:15,  7.59s/it]
Loading safetensors checkpoint shards:  36% Completed | 5/14 [00:37<01:06,  7.42s/it]
Loading safetensors checkpoint shards:  43% Completed | 6/14 [00:45<00:59,  7.49s/it]
Loading safetensors checkpoint shards:  50% Completed | 7/14 [00:52<00:52,  7.46s/it]
Loading safetensors checkpoint shards:  57% Completed | 8/14 [00:59<00:44,  7.42s/it]
Loading safetensors checkpoint shards:  64% Completed | 9/14 [01:07<00:37,  7.44s/it]
Loading safetensors checkpoint shards:  71% Completed | 10/14 [01:15<00:30,  7.55s/it]
Loading safetensors checkpoint shards:  79% Completed | 11/14

(EngineCore_DP0 pid=3759800) INFO 11-20 23:14:31 [default_loader.py:267] Loading weights took 100.56 seconds
(EngineCore_DP0 pid=3759800) INFO 11-20 23:14:32 [gpu_model_runner.py:2653] Model loading took 61.0375 GiB and 101.303527 seconds
(EngineCore_DP0 pid=3759800) INFO 11-20 23:14:46 [backends.py:548] Using cache directory: /home/tkavuru/.cache/vllm/torch_compile_cache/41f1b2542d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3759800) INFO 11-20 23:14:46 [backends.py:559] Dynamo bytecode transform time: 14.03 s
(EngineCore_DP0 pid=3759800) INFO 11-20 23:14:51 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 4.147 s
(EngineCore_DP0 pid=3759800) INFO 11-20 23:15:01 [monitor.py:34] torch.compile takes 14.03 s in total
(EngineCore_DP0 pid=3759800) INFO 11-20 23:15:02 [gpu_worker.py:298] Available KV cache memory: 8.56 GiB
(EngineCore_DP0 pid=3759800) INFO 11-20 23:15:02 [kv_cache_utils.py:1087] GPU KV cache size: 35,056 tokens


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:08<00:00,  7.95it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:03<00:00, 10.61it/s]


(EngineCore_DP0 pid=3759800) INFO 11-20 23:15:15 [gpu_model_runner.py:3480] Graph capturing finished in 12 secs, took 1.32 GiB
(EngineCore_DP0 pid=3759800) INFO 11-20 23:15:15 [core.py:210] init engine (profile, create kv cache, warmup model) took 43.02 seconds
INFO 11-20 23:15:16 [llm.py:306] Supported_tasks: ['generate']


In [9]:
# ==============================================================================
# THE "ADAPTIVE COMPILER" PROMPT (With Double Braces for Escaping)
# ==============================================================================

# NOTE: All C code braces are {{ }} so Python .format() ignores them.
# The only single braces are for {func_name}.
system_prompt = """You are an expert C Build Engineer & Test Architect.
Your task is to generate a `mock_context.h` header file.

### CORE PHILOSOPHY:
There may be **many** external dependencies, or there may be **none**.
- **Your Job:** Find exactly what is missing and define it.
- **Anti-Hallucination:** If an item (struct, function, macro) is NOT in the code, **DO NOT** generate it.

### IMPLEMENTATION STANDARDS (If found):

1. **FUNCTIONS (The Spy Pattern):**
    - **Goal:** We must be able to detect *if* a function was called and control *what* it returns.
    - **Requirement:** For every external function `foo` found:
        - Define `int g_cnt_foo = 0;` (Call Counter).
        - Define `int g_ret_foo = 0;` (Return Value, if non-void).
        - Implement stub: `TYPE foo(ARGS) {{ g_cnt_foo++; return g_ret_foo; }}`
    - **Signatures:** Infer return type and arguments strictly from the call site usage.

2. **STRUCTS (Field Inference):**
    - **Goal:** The code must compile.
    - **Requirement:** If code uses `ptr->x`, define `struct Name {{ int x; }};`.
    - **Nesting:** If code uses `ptr->sub.val`, define `struct Sub {{ int val; }};` and include it in the parent.

3. **MACROS & CONSTANTS:**
    - **Goal:** Valid values for comparison/assignment.
    - **Requirement:** `#define` all UPPERCASE identifiers found to safe defaults (e.g., 1, 0xFF).

4. **TYPEDEFS:**
    - **Goal:** Resolve non-standard types.
    - **Requirement:** `typedef` unknown scalars (e.g., `u8`) to standard types.

### EXCLUSIONS:
- **DO NOT** stub the function under test (`{func_name}`).
- **DO NOT** redefine standard types (`int`, `char`, `size_t`).

### OUTPUT:
Return **ONLY** the C code for `mock_context.h`.
"""

prompts = []
for _, row in full_df.iterrows():
    vuln_code = row['vuln_func_body']
    fixed_code = row.get('fixed_func_body', "") 
    func_name = row['func_name']

    # This .format() call will now work because C braces are escaped as {{ }}
    prompt_specific = system_prompt.format(func_name=func_name)

    user_msg = f"""
        **Function Under Test (DO NOT MOCK):** `{func_name}`

        **Source Code (Vulnerable + Fixed):**
        ```c
        {vuln_code}
        
        {fixed_code}
        ```

        **Task:**
        1. Scan the code for undefined identifiers (structs, functions, macros).
        2. **If NONE are found:** Return an empty header or just standard includes.
        3. **If FOUND:** Implement them in `mock_context.h` using the Spy Pattern.
        """
    
    messages = [
        {"role": "system", "content": prompt_specific},
        {"role": "user", "content": user_msg}
    ]
    prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

# Sampling
sampling_params = SamplingParams(temperature=0.1, max_tokens=4096)

print(f"-> Sending {len(prompts)} prompts to A100...")
outputs = llm.generate(prompts, sampling_params)

results = []
for i, output in enumerate(outputs):
    code_context = extract_code_block(output.outputs[0].text)
    
    row_data = full_df.iloc[i].to_dict()
    row_data['generated_context'] = code_context
    row_data['has_context'] = len(code_context) > 10
    results.append(row_data)

result_df = pd.DataFrame(results)
save_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
result_df.to_csv(save_path, index=False)

print(f"Saved processed data to: {save_path}")

-> Sending 71 prompts to A100...


Processed prompts: 100%|██████████| 71/71 [01:36<00:00,  1.35s/it, est. speed input: 805.63 toks/s, output: 168.07 toks/s] 

Saved processed data to: data_context/dataset_unified_with_context.csv


,func_name,fixed_func_idx,cve_list,cwe_list,vuln_func_body,fixed_func_body,context_norm,filter_reason,generated_context,has_context
0,hns_ppe_get_sset_count,80.0,['CVE-2017-18222'],['CWE-119'],int hns_ppe_get_sset_count(int stringset)\n{\n...,int hns_ppe_get_sset_count(int stringset)\n{\n...,"{'Execution Environment': [array([], dtype=obj...",The function performs a simple conditional che...,#ifndef MOCK_CONTEXT_H\n#define MOCK_CONTEXT_H...,True
1,shift_and_mask,526.0,['CVE-2024-44981'],['CWE-190'],static unsigned long shift_and_mask(unsigned l...,static unsigned long shift_and_mask(unsigned l...,"{'Execution Environment': [array([], dtype=obj...",The function performs simple bitwise operation...,#ifndef MOCK_CONTEXT_H\n\n#define MOCK_CONTEXT...,True
2,init_syntax_once,3262.0,['CVE-1999-0199'],['CWE-252'],init_syntax_once ()\n{\n register int c;\n ...,init_syntax_once ()\n{\n register int c;\n ...,"{'Execution Environment': [array([], dtype=obj...",The function performs observable logic within ...,#ifndef MOCK_CONTEXT_H\n#define MOCK_CONTEXT_H...,True
3,udf_translate_to_linux,4761.0,['CVE-2014-9731'],['CWE-17'],static int udf_translate_to_linux(uint8_t *new...,static int udf_translate_to_linux(uint8_t *new...,"{'Execution Environment': [array([], dtype=obj...",The function performs observable logic within ...,#ifndef MOCK_CONTEXT_H\n#define MOCK_CONTEXT_H...,True
4,atol8,6836.0,['CVE-2017-14166'],['CWE-125'],"atol8(const char *p, size_t char_cnt)\n{\n\tin...","atol8(const char *p, size_t char_cnt)\n{\n\tin...","{'Execution Environment': [array([], dtype=obj...",The function performs observable logic within ...,#ifndef MOCK_CONTEXT_H\n#define MOCK_CONTEXT_H...,True
